# Занятие 1. Пайплайн инвентаризации языковых датасетов

**Цель практики:** не OCR и не ASR, а стартовая карта проекта: какие языки берем в поле зрения и какие открытые данные уже можно найти.

В этой тетрадке мы собираем первоначальную информацию обычным воспроизводимым пайплайном:

1. берет редактируемый seed list основных живых языков народов России без диалектального уровня;
2. показывает, как выглядит ответ OPUS и какие поля из него достаем;
3. показывает, как выглядит карточка/ответ Hugging Face Datasets и какие поля из него достаем;
4. проверяет OPUS API на параллельные данные с русским;
5. проверяет Hugging Face Datasets как каталог опубликованных корпусов;
6. собирает таблицу, которую можно открыть в Google Sheets и дальше править руками.

Готовый снапшот этой таблицы уже создан в Google Sheets: https://docs.google.com/spreadsheets/d/1Qfr6JCB5CF-NLwQBODStqfhesrYw9tIVh2s_A6cg0d8

In [ ]:
!pip -q install pandas requests openpyxl

In [ ]:
import os, re, json, textwrap, math, statistics, random, io, time
from pathlib import Path
import pandas as pd
import numpy as np
import requests

DATA_DIR = Path('/content/lowres_lab')
DATA_DIR.mkdir(exist_ok=True)

def show_df(df, n=10):
    display(df.head(n))

def save_artifact(name, obj):
    path = DATA_DIR / name
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
    else:
        path.write_text(str(obj), encoding='utf-8')
    print('saved:', path)

from datetime import datetime, timezone
import re

## 1. Seed list языков

Это не “истина навсегда”, а стартовая рабочая рамка для курса. Ее надо обсуждать и уточнять: какие языки добавить, где объединять варианты, где наоборот нельзя смешивать разные языковые сообщества.

In [ ]:
LANGUAGES = [{'language_ru': 'татарский', 'language_en': 'Tatar', 'family': 'Тюркская', 'branch': 'кыпчакская', 'iso639_3': 'tat', 'opus_code': 'tt'}, {'language_ru': 'башкирский', 'language_en': 'Bashkir', 'family': 'Тюркская', 'branch': 'кыпчакская', 'iso639_3': 'bak', 'opus_code': 'ba'}, {'language_ru': 'чувашский', 'language_en': 'Chuvash', 'family': 'Тюркская', 'branch': 'огурская', 'iso639_3': 'chv', 'opus_code': 'chv'}, {'language_ru': 'якутский / саха', 'language_en': 'Sakha / Yakut', 'family': 'Тюркская', 'branch': 'сибирская', 'iso639_3': 'sah', 'opus_code': 'sah'}, {'language_ru': 'тувинский', 'language_en': 'Tuvan', 'family': 'Тюркская', 'branch': 'сибирская', 'iso639_3': 'tyv', 'opus_code': 'tyv'}, {'language_ru': 'хакасский', 'language_en': 'Khakas', 'family': 'Тюркская', 'branch': 'сибирская', 'iso639_3': 'kjh', 'opus_code': 'kjh'}, {'language_ru': 'алтайский', 'language_en': 'Altai', 'family': 'Тюркская', 'branch': 'сибирская', 'iso639_3': 'alt', 'opus_code': 'alt'}, {'language_ru': 'кумыкский', 'language_en': 'Kumyk', 'family': 'Тюркская', 'branch': 'кыпчакская', 'iso639_3': 'kum', 'opus_code': 'kum'}, {'language_ru': 'карачаево-балкарский', 'language_en': 'Karachay-Balkar', 'family': 'Тюркская', 'branch': 'кыпчакская', 'iso639_3': 'krc', 'opus_code': 'krc'}, {'language_ru': 'ногайский', 'language_en': 'Nogai', 'family': 'Тюркская', 'branch': 'кыпчакская', 'iso639_3': 'nog', 'opus_code': 'nog'}, {'language_ru': 'крымскотатарский', 'language_en': 'Crimean Tatar', 'family': 'Тюркская', 'branch': 'кыпчакско-огузская', 'iso639_3': 'crh', 'opus_code': 'crh'}, {'language_ru': 'удмуртский', 'language_en': 'Udmurt', 'family': 'Уральская', 'branch': 'пермская', 'iso639_3': 'udm', 'opus_code': 'udm'}, {'language_ru': 'коми-зырянский', 'language_en': 'Komi-Zyrian', 'family': 'Уральская', 'branch': 'пермская', 'iso639_3': 'kpv', 'opus_code': 'kpv'}, {'language_ru': 'коми-пермяцкий', 'language_en': 'Komi-Permyak', 'family': 'Уральская', 'branch': 'пермская', 'iso639_3': 'koi', 'opus_code': 'koi'}, {'language_ru': 'эрзянский', 'language_en': 'Erzya', 'family': 'Уральская', 'branch': 'мордовская', 'iso639_3': 'myv', 'opus_code': 'myv'}, {'language_ru': 'мокшанский', 'language_en': 'Moksha', 'family': 'Уральская', 'branch': 'мордовская', 'iso639_3': 'mdf', 'opus_code': 'mdf'}, {'language_ru': 'марийский луговой', 'language_en': 'Meadow Mari', 'family': 'Уральская', 'branch': 'марийская', 'iso639_3': 'mhr', 'opus_code': 'mhr'}, {'language_ru': 'марийский горный', 'language_en': 'Hill Mari', 'family': 'Уральская', 'branch': 'марийская', 'iso639_3': 'mrj', 'opus_code': 'mrj'}, {'language_ru': 'карельский', 'language_en': 'Karelian', 'family': 'Уральская', 'branch': 'прибалтийско-финская', 'iso639_3': 'krl', 'opus_code': 'krl'}, {'language_ru': 'вепсский', 'language_en': 'Veps', 'family': 'Уральская', 'branch': 'прибалтийско-финская', 'iso639_3': 'vep', 'opus_code': 'vep'}, {'language_ru': 'хантыйский', 'language_en': 'Khanty', 'family': 'Уральская', 'branch': 'угорская', 'iso639_3': 'kca', 'opus_code': None}, {'language_ru': 'мансийский', 'language_en': 'Mansi', 'family': 'Уральская', 'branch': 'угорская', 'iso639_3': 'mns', 'opus_code': 'mns'}, {'language_ru': 'ненецкий', 'language_en': 'Nenets', 'family': 'Уральская', 'branch': 'самодийская', 'iso639_3': 'yrk', 'opus_code': 'yrk'}, {'language_ru': 'чеченский', 'language_en': 'Chechen', 'family': 'Северокавказская', 'branch': 'нахская', 'iso639_3': 'che', 'opus_code': 'ce'}, {'language_ru': 'ингушский', 'language_en': 'Ingush', 'family': 'Северокавказская', 'branch': 'нахская', 'iso639_3': 'inh', 'opus_code': 'inh'}, {'language_ru': 'аварский', 'language_en': 'Avar', 'family': 'Северокавказская', 'branch': 'нахско-дагестанская', 'iso639_3': 'ava', 'opus_code': 'av'}, {'language_ru': 'даргинский', 'language_en': 'Dargwa', 'family': 'Северокавказская', 'branch': 'нахско-дагестанская', 'iso639_3': 'dar', 'opus_code': 'dar'}, {'language_ru': 'лезгинский', 'language_en': 'Lezgian', 'family': 'Северокавказская', 'branch': 'нахско-дагестанская', 'iso639_3': 'lez', 'opus_code': 'lez'}, {'language_ru': 'лакский', 'language_en': 'Lak', 'family': 'Северокавказская', 'branch': 'нахско-дагестанская', 'iso639_3': 'lbe', 'opus_code': 'lbe'}, {'language_ru': 'рутульский', 'language_en': 'Rutul', 'family': 'Северокавказская', 'branch': 'нахско-дагестанская', 'iso639_3': 'rut', 'opus_code': 'rut'}, {'language_ru': 'адыгейский', 'language_en': 'Adyghe', 'family': 'Северокавказская', 'branch': 'абхазо-адыгская', 'iso639_3': 'ady', 'opus_code': 'ady'}, {'language_ru': 'кабардино-черкесский', 'language_en': 'Kabardian', 'family': 'Северокавказская', 'branch': 'абхазо-адыгская', 'iso639_3': 'kbd', 'opus_code': 'kbd'}, {'language_ru': 'абазинский', 'language_en': 'Abaza', 'family': 'Северокавказская', 'branch': 'абхазо-адыгская', 'iso639_3': 'abq', 'opus_code': None}, {'language_ru': 'бурятский', 'language_en': 'Buryat', 'family': 'Монгольская', 'branch': 'монгольская', 'iso639_3': 'bxr', 'opus_code': 'bxr'}, {'language_ru': 'калмыцкий', 'language_en': 'Kalmyk', 'family': 'Монгольская', 'branch': 'ойратская', 'iso639_3': 'xal', 'opus_code': 'xal'}, {'language_ru': 'эвенкийский', 'language_en': 'Evenki', 'family': 'Тунгусо-маньчжурская', 'branch': 'тунгусская', 'iso639_3': 'evn', 'opus_code': 'evn'}, {'language_ru': 'нанайский', 'language_en': 'Nanai', 'family': 'Тунгусо-маньчжурская', 'branch': 'тунгусская', 'iso639_3': 'gld', 'opus_code': 'gld'}, {'language_ru': 'нивхский', 'language_en': 'Nivkh', 'family': 'изолят / палеоазиатская группа', 'branch': 'нивхская', 'iso639_3': 'niv', 'opus_code': None}, {'language_ru': 'чукотский', 'language_en': 'Chukchi', 'family': 'чукотско-камчатская', 'branch': 'чукотская', 'iso639_3': 'ckt', 'opus_code': None}, {'language_ru': 'корякский', 'language_en': 'Koryak', 'family': 'чукотско-камчатская', 'branch': 'чукотская', 'iso639_3': 'kpy', 'opus_code': None}, {'language_ru': 'алеутский', 'language_en': 'Aleut', 'family': 'эскимосско-алеутская', 'branch': 'алеутская', 'iso639_3': 'ale', 'opus_code': 'ale'}, {'language_ru': 'эскимосский / юпик', 'language_en': 'Yupik', 'family': 'эскимосско-алеутская', 'branch': 'эскимосская', 'iso639_3': 'ess', 'opus_code': None}]

seed_df = pd.DataFrame(LANGUAGES)
display(seed_df.groupby(['family', 'branch']).size().reset_index(name='languages'))
display(seed_df.head(12))

## 2. Источники пайплайна: OPUS API и Hugging Face API

Здесь мы работаем со структурированными источниками готовых датасетов. Это важное ограничение: API дают воспроизводимые поля и ссылки, а веб-поиск дает только кандидатов, которые потом нужно проверять человеком.

В этом занятии мы не используем Wikipedia: это хороший источник текстовых данных, но не каталог готовых датасетов. Сейчас нас интересует именно инвентаризация уже опубликованных датасетов и корпусов.

In [ ]:
OPUS_API = 'https://opus.nlpl.eu/opusapi'
HF_DATASETS_API = 'https://huggingface.co/api/datasets'

def api_get(url, params, attempts=3, timeout=30):
    """Загружает JSON из публичного API с повторами и паузами при rate limit."""
    last_error = None
    for attempt in range(attempts):
        try:
            r = requests.get(url, params=params, timeout=timeout, headers={'User-Agent': 'lowres-course-dataset-scout/1.0'})
            if r.status_code == 429 and attempt < attempts - 1:
                wait = int(r.headers.get('Retry-After', 2 + attempt * 2))
                print('rate limit, wait', wait, 'sec')
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r.json()
        except Exception as exc:
            last_error = exc
            if attempt < attempts - 1:
                time.sleep(1 + attempt * 2)
    raise last_error

def as_int(value):
    """Преобразует числовые поля OPUS в int, считая пустые значения нулем."""
    if value in ('', None):
        return 0
    return int(value)

def show_json_fragment(obj, keys=None, limit=1600):
    """Печатает небольшой фрагмент JSON, чтобы глазами увидеть форму ответа API."""
    if keys and isinstance(obj, dict):
        obj = {key: obj.get(key) for key in keys}
    text = json.dumps(obj, ensure_ascii=False, indent=2)
    print(text[:limit] + ('\n...' if len(text) > limit else ''))

def opus_pair_page_url(source, target, corpus='Tatoeba'):
    """Собирает ссылку на человеческую страницу OPUS для корпуса и языковой пары."""
    return f'https://opus.nlpl.eu/datasets/{corpus}?hi={source}&pair={target}'

def opus_pair_api_url(source, target):
    """Собирает ссылку на API-запрос OPUS для языковой пары."""
    return f'{OPUS_API}?source={source}&target={target}&preprocessing=xml&version=latest'

def hf_dataset_page_url(dataset_id):
    """Собирает ссылку на карточку датасета на Hugging Face."""
    return f'https://huggingface.co/datasets/{dataset_id}'

def hf_dataset_api_url(dataset_id):
    """Собирает ссылку на API-ответ Hugging Face по одному датасету."""
    return f'{HF_DATASETS_API}/{dataset_id}'

## 3. Пример OPUS: страница пары, API-ответ и извлекаемые поля

Возьмем пару `ru-udm`: русский и удмуртский. У OPUS есть человеческие страницы корпусов и API. Для пайплайна важнее API, но страница нужна, чтобы человек мог быстро открыть источник и проверить контекст: корпус, лицензию, форматы скачивания, предупреждения OPUS.

In [ ]:
OPUS_EXAMPLE = {
    'source': 'ru',
    'target': 'udm',
    'human_page': opus_pair_page_url('ru', 'udm', corpus='Tatoeba'),
    'api_url': opus_pair_api_url('ru', 'udm'),
}
OPUS_EXAMPLE

In [ ]:
# OPUS иногда отвечает медленно. Если API не ответил за короткое время,
# используем сохраненный пример той же структуры, чтобы занятие не зависло.
OPUS_FALLBACK = {
    'corpora': [
        {
            'corpus': 'Tatoeba',
            'source': 'ru',
            'target': 'udm',
            'alignment_pairs': '337',
            'documents': '1',
            'latest': 'True',
            'preprocessing': 'xml',
            'version': 'latest',
            'url': 'https://opus.nlpl.eu/Tatoeba.php',
        },
    ]
}

try:
    opus_raw = api_get(OPUS_API, {
        'source': OPUS_EXAMPLE['source'],
        'target': OPUS_EXAMPLE['target'],
        'preprocessing': 'xml',
        'version': 'latest',
    }, attempts=1, timeout=8)
    opus_raw_source = 'live OPUS API'
except Exception as exc:
    print('OPUS API сейчас не ответил быстро:', exc)
    opus_raw = OPUS_FALLBACK
    opus_raw_source = 'fallback example'

print('Источник примера:', opus_raw_source)
print('Страница пары/корпуса для человека:', OPUS_EXAMPLE['human_page'])
print('API URL для пайплайна:', OPUS_EXAMPLE['api_url'])
show_json_fragment(opus_raw, keys=['corpora'], limit=2200)

In [ ]:
opus_rows = pd.DataFrame(opus_raw.get('corpora', []))
opus_fields_we_extract = opus_rows[[
    'corpus',
    'source',
    'target',
    'alignment_pairs',
    'documents',
    'preprocessing',
    'version',
]].copy()
display(opus_fields_we_extract)

print('Что пайплайн кладет в итоговую таблицу:')
display(pd.DataFrame([{
    'opus_ru_parallel_pairs': opus_fields_we_extract['alignment_pairs'].map(as_int).sum(),
    'opus_ru_parallel_documents': opus_fields_we_extract['documents'].map(as_int).sum(),
    'opus_ru_parallel_corpora': '; '.join(
        f"{row.corpus} ({row.alignment_pairs})"
        for row in opus_fields_we_extract.itertuples()
    ),
    'parallel_with_russian_source': OPUS_EXAMPLE['api_url'],
}]))

## 4. Пример Hugging Face: карточка датасета, API-ответ и извлекаемые поля

На Hugging Face у каждого датасета есть страница-карточка и API-ответ. Страница нужна человеку: посмотреть README, лицензию, файлы, ограничения доступа. API нужен пайплайну: собрать id, теги языка, размер, число примеров, downloads и признаки параллельности.

In [ ]:
HF_EXAMPLE_ID = 'udmurtNLP/flores-250-rus-udm'
HF_EXAMPLE = {
    'dataset_id': HF_EXAMPLE_ID,
    'human_page': hf_dataset_page_url(HF_EXAMPLE_ID),
    'api_url': hf_dataset_api_url(HF_EXAMPLE_ID),
}
HF_EXAMPLE

In [ ]:
hf_raw = api_get(HF_DATASETS_API + '/' + HF_EXAMPLE_ID, {}, attempts=2, timeout=20)

print('Страница датасета для человека:', HF_EXAMPLE['human_page'])
print('API URL для пайплайна:', HF_EXAMPLE['api_url'])
show_json_fragment(hf_raw, keys=['id', 'tags', 'downloads', 'likes', 'cardData', 'siblings'], limit=2600)

In [ ]:
card_data = hf_raw.get('cardData') or {}
dataset_info = card_data.get('dataset_info') or {}
splits = dataset_info.get('splits') or []
features = dataset_info.get('features') or []

hf_fields_we_extract = {
    'hf_dataset_id': hf_raw.get('id'),
    'hf_page': HF_EXAMPLE['human_page'],
    'hf_downloads': hf_raw.get('downloads'),
    'hf_likes': hf_raw.get('likes'),
    'hf_language_tags': '; '.join(tag for tag in hf_raw.get('tags', []) if tag.startswith('language:')),
    'hf_size_categories': '; '.join(tag.replace('size_categories:', '') for tag in hf_raw.get('tags', []) if tag.startswith('size_categories:')),
    'hf_splits': '; '.join(f"{s.get('name')} ({s.get('num_examples')} examples)" for s in splits),
    'hf_features': '; '.join(f"{f.get('name')}:{f.get('dtype')}" for f in features),
    'hf_files': '; '.join(s.get('rfilename', '') for s in hf_raw.get('siblings', [])[:5]),
}
display(pd.DataFrame([hf_fields_we_extract]).T.rename(columns={0: 'value'}))

## 5. Почему здесь не нужны агенты

Эту задачу лучше решать обычным алгоритмом. У нас есть фиксированный список языков, заранее известные API, понятные поля ответа и воспроизводимые шаги обработки. Если нужно проверить конкретную языковую пару или скачать конкретный датасет, это тоже проще, надежнее и дешевле сделать обычным кодом или руками.

Агенты становятся уместны в другой ситуации: когда вход неформализован, источники заранее неизвестны, а набор решений нельзя полностью подготовить до запуска. Например: “найди все пригодные материалы для коми-пермяцкого, не перепутай его с коми-зырянским, отдели готовые датасеты от просто текстовых источников, оцени лицензионные риски и предложи, что проверить человеку”. Это уже не чистая табличная инвентаризация, а исследовательская разведка с неоднозначными решениями.

В этой тетрадке мы сознательно оставляем только пайплайн сбора датасетов. Это хороший пример того, где агент не нужен.

## 6. Функции свертки источников в наблюдения

In [ ]:
def query_opus_for_language(opus_code):
    """Собирает сводку OPUS по моноязычным и русско-параллельным данным языка."""
    empty = {
        'opus_ru_parallel_pairs': 0,
        'opus_ru_parallel_documents': 0,
        'opus_ru_parallel_corpora': '',
        'opus_mono_pairs_or_segments': 0,
        'opus_mono_documents': 0,
        'opus_mono_corpora': '',
    }
    if not opus_code:
        return {'opus_checked': False, **empty}
    try:
        data = api_get(OPUS_API, {
            'source': 'ru',
            'target': opus_code,
            'preprocessing': 'xml',
            'version': 'latest',
        }, attempts=1, timeout=8)
    except Exception as exc:
        return {'opus_checked': False, 'opus_error': str(exc), **empty}
    corpora = data.get('corpora', [])
    parallel = [c for c in corpora if {c.get('source'), c.get('target')} == {'ru', opus_code}]
    mono = [c for c in corpora if c.get('source') == opus_code and not c.get('target')]
    return {
        'opus_checked': True,
        'opus_ru_parallel_pairs': sum(as_int(c.get('alignment_pairs')) for c in parallel),
        'opus_ru_parallel_documents': sum(as_int(c.get('documents')) for c in parallel),
        'opus_ru_parallel_corpora': '; '.join(f"{c.get('corpus')} ({c.get('alignment_pairs') or 0})" for c in parallel),
        'opus_mono_pairs_or_segments': sum(as_int(c.get('alignment_pairs')) for c in mono),
        'opus_mono_documents': sum(as_int(c.get('documents')) for c in mono),
        'opus_mono_corpora': '; '.join(f"{c.get('corpus')} ({c.get('alignment_pairs') or 0})" for c in mono),
    }

def query_huggingface_for_language(row):
    """Ищет датасеты на Hugging Face и собирает сводку вероятных ресурсов языка.

    HF-поиск обычно не дает точного количества документов или предложений.
    Поэтому сохраняем разведочные метаданные: число кандидатов, вероятные русско-
    параллельные датасеты, топ id датасетов, скачивания и категории размера из тегов.
    """
    language_tags = {
        f"language:{row.get('opus_code')}" if row.get('opus_code') else '',
        f"language:{row.get('iso639_3')}" if row.get('iso639_3') else '',
    }
    language_tags.discard('')
    search_terms = [row.get('language_en'), row.get('language_ru')]
    seen = {}
    attempted = 0
    successful = 0
    for tag in language_tags:
        attempted += 1
        try:
            results = api_get(HF_DATASETS_API, {'filter': tag, 'limit': 10}, attempts=2, timeout=20)
        except Exception:
            continue
        if not isinstance(results, list):
            continue
        successful += 1
        for dataset in results:
            dataset_id = dataset.get('id')
            if dataset_id:
                seen[dataset_id] = dataset

    for term in [x for x in search_terms if x]:
        attempted += 1
        try:
            results = api_get(HF_DATASETS_API, {'search': term, 'limit': 10}, attempts=2, timeout=20)
        except Exception:
            continue
        if not isinstance(results, list):
            continue
        successful += 1
        for dataset in results:
            dataset_id = dataset.get('id')
            if dataset_id:
                seen[dataset_id] = dataset

    lang_texts = [
        str(row.get('language_en', '')).lower(),
        str(row.get('language_ru', '')).lower(),
    ]

    def mentions_language_name(text, names):
        """Проверяет, встречается ли полное название языка как отдельная фраза."""
        for name in names:
            if not name:
                continue
            for part in re.split(r'\s*/\s*|\s+-\s+', name):
                part = part.strip()
                if len(part) >= 4 and re.search(rf'(?<![\w-]){re.escape(part)}(?![\w-])', text):
                    return True
        return False

    datasets = []
    for dataset in seen.values():
        tags = set(dataset.get('tags') or [])
        haystack = ' '.join([
            dataset.get('id', ''),
            dataset.get('description', '') or '',
            ' '.join(tags),
        ]).lower()
        tagged = bool(tags & language_tags)
        mentioned = mentions_language_name(haystack, lang_texts)
        if tagged or mentioned:
            datasets.append(dataset)

    def is_ru_parallel(dataset):
        """Эвристически определяет, похож ли HF-датасет на русско-параллельный."""
        tags = set(dataset.get('tags') or [])
        text = ' '.join([
            dataset.get('id', ''),
            dataset.get('description', '') or '',
            ' '.join(tags),
        ]).lower()
        return (
            'language:ru' in tags
            or 'russian' in text
            or 'рус' in text
            or '-rus-' in text
            or 'rus-' in text
        )

    def specificity_score(dataset):
        """Ставит языково-специфичные датасеты выше широких многоязычных коллекций."""
        tags = set(dataset.get('tags') or [])
        text = ' '.join([
            dataset.get('id', ''),
            dataset.get('description', '') or '',
        ]).lower()
        language_tag_count = sum(1 for tag in tags if tag.startswith('language:'))
        if mentions_language_name(text, lang_texts):
            return 2
        if language_tag_count <= 5:
            return 1
        return 0

    top = sorted(
        datasets,
        key=lambda d: (specificity_score(d), d.get('downloads') or 0),
        reverse=True,
    )[:5]
    size_categories = sorted({
        tag.replace('size_categories:', '')
        for dataset in datasets
        for tag in (dataset.get('tags') or [])
        if tag.startswith('size_categories:')
    })
    return {
        'hf_checked': successful > 0,
        'hf_query_attempts': attempted,
        'hf_query_successes': successful,
        'hf_dataset_count': len(datasets),
        'hf_ru_parallel_candidates': sum(1 for dataset in datasets if is_ru_parallel(dataset)),
        'hf_top_datasets': '; '.join(dataset.get('id', '') for dataset in top),
        'hf_downloads_sum': sum(int(dataset.get('downloads') or 0) for dataset in datasets),
        'hf_size_categories': '; '.join(size_categories),
        'hf_source_url': 'https://huggingface.co/datasets',
    }

## 7. Запуск пайплайна по всем языкам

In [ ]:
def scout_language(row):
    """Собирает все наблюдения по одному языку в одну сериализуемую строку."""
    observation = dict(row)
    observation.update(query_opus_for_language(row.get('opus_code')))
    observation.update(query_huggingface_for_language(row))
    observation['parallel_with_russian_source'] = 'https://opus.nlpl.eu/opusapi'
    observation['monolingual_source'] = 'OPUS monolingual rows; Hugging Face dataset search'
    observation['checked_at_utc'] = datetime.now(timezone.utc).strftime('%Y-%m-%d')
    return observation

def run_dataset_inventory_pipeline(languages):
    """Запускает воспроизводимый пайплайн инвентаризации датасетов."""
    result = {
        'sources': ['OPUS API', 'Hugging Face dataset API'],
        'languages_total': len(languages),
        'observations': [],
        'errors': [],
    }
    for i, lang in enumerate(languages, 1):
        print(f"[{i}/{len(languages)}] {lang['language_ru']}")
        try:
            result['observations'].append(scout_language(lang))
        except Exception as exc:
            result['errors'].append({'language_ru': lang['language_ru'], 'error': str(exc)})
    inventory = pd.DataFrame(result['observations'])
    inventory = inventory.sort_values(['family', 'branch', 'language_ru']).reset_index(drop=True)
    result['inventory'] = inventory
    result['summary'] = {
        'languages_total': len(languages),
        'languages_checked': len(inventory),
        'with_opus_ru_parallel': int((inventory['opus_ru_parallel_pairs'] > 0).sum()),
        'with_hf_candidates': int((inventory['hf_dataset_count'] > 0).sum()),
        'errors': len(result['errors']),
    }
    return result

pipeline_result = run_dataset_inventory_pipeline(LANGUAGES)
pipeline_result['summary']

In [ ]:
inventory = pipeline_result['inventory']
display(inventory.head(20))
display(inventory.groupby('family')[['opus_ru_parallel_pairs', 'opus_mono_pairs_or_segments']].sum().sort_values('opus_ru_parallel_pairs', ascending=False))

save_artifact('lesson01_language_dataset_inventory.csv', inventory)
save_artifact('lesson01_dataset_inventory_summary.json', json.dumps(pipeline_result['summary'], ensure_ascii=False, indent=2))

## 8. Google Sheets

На занятии можно открыть готовый Google Sheet и править его как общий рабочий артефакт:

https://docs.google.com/spreadsheets/d/1Qfr6JCB5CF-NLwQBODStqfhesrYw9tIVh2s_A6cg0d8

В Colab эта тетрадка сохраняет CSV в `/content/lowres_lab/lesson01_language_dataset_inventory.csv`. Его можно загрузить в Google Sheets или использовать как основу для обновления общей таблицы.

## 9. Как автоматизировать обновление

Разовый запуск полезен для старта, но карта датасетов быстро устаревает: в OPUS появляются новые релизы, в Hugging Face загружают корпуса, национальные проекты открывают новые таблицы, а часть ссылок ломается.

Для этого нужен регулярный фоновый пайплайн:

1. **Scheduler** запускает пайплайн по расписанию: например, раз в неделю или раз в месяц.
2. **Collector** заново обходит источники готовых датасетов: OPUS, Hugging Face, GitHub-релизы, национальные корпуса, каталоги открытых данных, архивы с опубликованными корпусами.
3. **State store** хранит предыдущий снимок таблицы: CSV в GitHub, Google Sheet, SQLite или маленький JSON.
4. **Diff checker** сравнивает старую и новую версии: новые языки, новые корпуса, рост/падение counts, ошибки API.
5. **Updater** обновляет Google Sheet только для безопасных полей: counts, даты проверки, ссылки на источники.
6. **Human review** получает спорные изменения: новый источник без понятной лицензии, резкое падение counts, объединение языков/вариантов, изменение классификации.

Где здесь веб-поиск? Его можно добавить отдельным ручным или полуавтоматическим слоем обнаружения: запросы вроде `"удмуртский корпус скачать"`, `"Udmurt dataset"`, `"site:github.com udmurt corpus"`, `"site:huggingface.co/datasets udmurt"`. Но веб-поиск лучше использовать как слой кандидатов, а не как источник финальных чисел. Найденные ссылки должны попадать в лист `candidates_for_review`, пока человек не подтвердит язык, лицензию, формат, объем и надежность источника.

Самый простой стек для курса:

- `scripts/build_language_dataset_inventory.py` лежит в GitHub;
- GitHub Actions запускает его по cron;
- скрипт сохраняет новый CSV;
- отдельный шаг через Google Sheets API обновляет таблицу;
- если diff большой или появились ошибки, workflow создает issue/комментарий для ручной проверки.

Colab для такого расписания не подходит: он хорош для занятия и ручного запуска, но не для надежного фонового мониторинга.

In [ ]:
def compare_inventory_snapshots(old_df, new_df):
    """Возвращает изменения по строкам между двумя снимками инвентаризации."""
    key = 'iso639_3'
    old = old_df.set_index(key)
    new = new_df.set_index(key)
    rows = []

    for code in sorted(set(old.index) | set(new.index)):
        if code not in old.index:
            rows.append({'iso639_3': code, 'change_type': 'new_language', 'needs_human_review': True})
            continue
        if code not in new.index:
            rows.append({'iso639_3': code, 'change_type': 'missing_language', 'needs_human_review': True})
            continue

        old_pairs = int(old.loc[code, 'opus_ru_parallel_pairs'])
        new_pairs = int(new.loc[code, 'opus_ru_parallel_pairs'])
        delta = new_pairs - old_pairs
        if delta != 0:
            rows.append({
                'iso639_3': code,
                'language_ru': new.loc[code, 'language_ru'],
                'change_type': 'parallel_count_changed',
                'old_pairs': old_pairs,
                'new_pairs': new_pairs,
                'delta': delta,
                'needs_human_review': abs(delta) > max(1000, old_pairs * 0.5),
            })

    return pd.DataFrame(rows)

# Мини-демо: имитируем, что через месяц OPUS нашел больше параллельных предложений для удмуртского.
old_snapshot = inventory.copy()
new_snapshot = inventory.copy()
new_snapshot.loc[new_snapshot['iso639_3'] == 'udm', 'opus_ru_parallel_pairs'] += 250

diff = compare_inventory_snapshots(old_snapshot, new_snapshot)
display(diff)
save_artifact('lesson01_inventory_diff_demo.csv', diff)

### Пример GitHub Actions расписания

```yaml
name: update-language-dataset-inventory

on:
  schedule:
    - cron: "0 6 1 * *"  # 1 числа каждого месяца
  workflow_dispatch:

jobs:
  update:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install pandas openpyxl requests
      - run: python scripts/build_language_dataset_inventory.py
      - name: Update Google Sheet
        run: python scripts/update_google_sheet.py
        env:
          GOOGLE_SERVICE_ACCOUNT_JSON: ${{ secrets.GOOGLE_SERVICE_ACCOUNT_JSON }}
          SPREADSHEET_ID: "1Qfr6JCB5CF-NLwQBODStqfhesrYw9tIVh2s_A6cg0d8"
```

В реальном проекте секреты Google API нельзя хранить в notebook. Их кладут в GitHub Secrets, Google Cloud Secret Manager или другой защищенный secret store.

## Вопросы для отчета

1. Какие языковые семьи в таблице оказываются лучше всего покрыты параллельными данными с русским?
2. Где есть Википедия, но почти нет параллельных данных?
3. Где OPUS показывает нули: это значит “данных нет” или “мы не нашли правильный код/источник”?
4. Какие источники надо добавить следующими: национальные корпуса, сайты СМИ, библиотеки, архивы, Hugging Face, GitHub?
5. Какие результаты HF-поиска выглядят полезными, но требуют ручной проверки?
6. Какие веб-поисковые запросы вы бы добавили для своего языка?
7. Какие поля можно обновлять автоматически, а какие требуют human review?
8. Как часто стоит запускать фоновое обновление для такой таблицы и почему?